VII.) Autism-gene enrichment among early human–chimpanzee PFC expression differences

The goal is to test whether SFARI autism-associated genes are overrepresented among the 140 genes selected in the pylimma analysis.
We use two gene sets: SFARI Category 1 and SFARI Categories 1 + 2.

1.) Load packages

In [5]:
import pandas as pd
import numpy as np

2.) Download autism gene database SFARI with category 1 and category 2 genes

In [6]:

#Read SFARI human genes from 2026
sfari_all = pd.read_csv("data/sfari_human_genes_2026Q2.csv")

# Standardize the relevant columns
sfari_all = sfari_all.rename(columns={"gene-symbol": "Gene"})
sfari_all["Gene"] = sfari_all["Gene"].astype(str).str.strip().str.upper()
sfari_all["gene-score"] = pd.to_numeric(
    sfari_all["gene-score"], errors="coerce"
)

# Category 1: high-confidence autism genes
sfari_autism_category1 = (
    sfari_all.loc[sfari_all["gene-score"] == 1]
    .drop_duplicates("Gene")
    .sort_values("Gene")
    .reset_index(drop=True)
)

# Categories 1 and 2: high-confidence + strong-candidate genes
sfari_autism_category1_2 = (
    sfari_all.loc[sfari_all["gene-score"].isin([1, 2])]
    .drop_duplicates("Gene")
    .sort_values("Gene")
    .reset_index(drop=True)
)

# Save the derived analysis gene sets as results, not original input data.
sfari_autism_category1.to_csv(
    "results/sfari_autism_category1.csv",
    index=False
)

sfari_autism_category1_2.to_csv(
    "results/sfari_autism_category1_2.csv",
    index=False
)

print("All scored and unscored SFARI genes:", len(sfari_all))
print("SFARI Category 1 genes:", len(sfari_autism_category1))
print("SFARI Category 1 or 2 genes:", len(sfari_autism_category1_2))

All scored and unscored SFARI genes: 1289
SFARI Category 1 genes: 245
SFARI Category 1 or 2 genes: 957


3.) Match SFARI genes with the pylimma results

Both files contain gene symbols. Remove surrounding spaces and use uppercase symbols for matching. The background consists of all 11,667 genes tested by pylimma, not all genes in the genome. SFARI genes absent from the tested set cannot contribute to the Fisher tests. Gene aliases are not automatically converted; check unmatched symbols if more precise matching is needed.

In [7]:
pylimma_results = pd.read_csv("results/early_human_chimp_pylimma_results_human_spec.csv")

pylimma_results["Gene"] = pylimma_results["Gene"].astype(str).str.strip().str.upper()
pylimma_results["selected"] = (
    pylimma_results["selected"].astype(str).str.strip().str.lower().isin(["true", "1"])
)

if pylimma_results["Gene"].duplicated().any():
    raise ValueError("Gene symbols are duplicated in the pylimma results.")

tested_genes = set(pylimma_results["Gene"])
selected_genes = set(pylimma_results.loc[pylimma_results["selected"], "Gene"])

category1_genes = set(sfari_autism_category1["Gene"].astype(str).str.strip().str.upper())
category1_2_genes = set(sfari_autism_category1_2["Gene"].astype(str).str.strip().str.upper())

print("Genes tested:", len(tested_genes))
print("Genes selected:", len(selected_genes))
print("SFARI Category 1 genes among tested:", len(category1_genes & tested_genes))
print("SFARI Categories 1 + 2 genes among tested:", len(category1_2_genes & tested_genes))
print("SFARI Category 1 symbols not in tested genes:", len(category1_genes - tested_genes))
print("SFARI Categories 1 + 2 symbols not in tested genes:", len(category1_2_genes - tested_genes))

assert len(tested_genes) == 11667, "Unexpected number of tested genes."
assert len(selected_genes) == 140, "Unexpected number of selected genes."
assert selected_genes <= tested_genes

Genes tested: 11667
Genes selected: 140
SFARI Category 1 genes among tested: 217
SFARI Categories 1 + 2 genes among tested: 732
SFARI Category 1 symbols not in tested genes: 28
SFARI Categories 1 + 2 symbols not in tested genes: 225


4.) Fisher exact tests

Each 2 × 2 table compares the 140 selected genes with the other tested genes. The columns distinguish genes in the SFARI set from genes outside it. The one-sided Fisher test (alternative="greater") checks for overrepresentation in the selected genes.

In [8]:
from scipy.stats import fisher_exact
from IPython.display import display


def fisher_table(sfari_gene_set, title):
    autism_genes = sfari_gene_set & tested_genes
    other_genes = tested_genes - selected_genes

    table = pd.DataFrame(
        [
            [len(selected_genes & autism_genes), len(selected_genes - autism_genes)],
            [len(other_genes & autism_genes), len(other_genes - autism_genes)],
        ],
        index=["Selected (140)", "Not selected"],
        columns=["SFARI gene", "Not SFARI gene"],
    )
    odds_ratio, p_value = fisher_exact(table.to_numpy(), alternative="greater")

    print(title)
    display(table)
    print(f"Fisher exact test (one-sided overrepresentation): p = {p_value:.6g}")
    print(f"Odds ratio = {odds_ratio:.4g}")
    print()
    return table, p_value


table_cat1, p_cat1 = fisher_table(category1_genes, "SFARI Category 1")
table_cat1_2, p_cat1_2 = fisher_table(category1_2_genes, "SFARI Categories 1 + 2")

# Save the statistical outputs to results/ as well as displaying them.
table_cat1.to_csv("results/sfari_fisher_category1_table.csv")
table_cat1_2.to_csv("results/sfari_fisher_category1_2_table.csv")
pd.DataFrame({
    "sfari_category": ["Category 1", "Categories 1 + 2"],
    "selected_sfari_genes": [table_cat1.iloc[0, 0], table_cat1_2.iloc[0, 0]],
    "total_tested_sfari_genes": [table_cat1.iloc[:, 0].sum(), table_cat1_2.iloc[:, 0].sum()],
    "one_sided_fisher_p": [p_cat1, p_cat1_2],
}).to_csv("results/sfari_fisher_test_summary.csv", index=False)

SFARI Category 1


,SFARI gene,Not SFARI gene
Selected (140),0,140
Not selected,217,11310


Fisher exact test (one-sided overrepresentation): p = 1
Odds ratio = 0

SFARI Categories 1 + 2


,SFARI gene,Not SFARI gene
Selected (140),4,136
Not selected,728,10799


Fisher exact test (one-sided overrepresentation): p = 0.978922
Odds ratio = 0.4363



5.) Interpretation

A small one-sided p-value suggests that the proportion of SFARI genes is higher among the 140 selected genes than among the other tested genes. An odds ratio above 1 corresponds to enrichment. These are two related exploratory tests; the displayed p-values are not adjusted for performing both tests. Exact gene-symbol matching may miss historical aliases, so check unmatched identifiers before interpreting borderline results.